In [ ]:
from sympy.physics.units import temperature
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

In [2]:
#loading env, keys
load_dotenv()

#laoding llm model
# llm = ChatOllama(model="llama3.1:8b")
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",temperature=0.2)

In [ ]:
# Document Laoder
video_id = "7ARBJQn6QkM"
yt_api =YouTubeTranscriptApi()

transcript = yt_api.fetch(video_id)
trans = [doc.text for doc in transcript]
print(trans)

#Cleaning the transcript
transcript_clean = " ".join(doc.text for doc in transcript)
transcript_clean

In [4]:
# Text Splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
data_chunks = splitter.split_text(transcript_clean)
len(data_chunks)

118

In [5]:
# Convert to Embeddings -> Call embedding model
embeddings = OllamaEmbeddings(model='embeddinggemma')

In [6]:
# Vector store creation and storing
vc_store = FAISS.from_texts(data_chunks,embeddings)

In [8]:
# Retriever
retriever = vc_store.as_retriever(search_kwrgs={"k":3})

In [10]:
#Augementation
#Prompt Template

prompt = PromptTemplate(template="""You are a helpful Youtube brief assistant, Answer from the following context, If the context is insufficient then directly convey that I dont know. {context}, Here is the
Questions{query}""",input_variables=['context',"query"])

In [11]:
#Query
query = "How AI can change future generation?"

In [12]:
#creating Context
context = retriever.invoke(query)
#cleaning the context
cleaned_context = " ".join(i.page_content for i in context)

In [13]:
#Creating final prompt
final_prompt = prompt.invoke({'query':query,'context':cleaned_context})

In [14]:
#Generation
answer = llm.invoke(final_prompt)

In [15]:
answer.content[0]['text']

'Based on the context provided, AI can change the future generation by:\n\n* Allowing people to ask how they can use AI to do their jobs better (whether as a lawyer, doctor, chemist, or biologist).\n* Empowering individuals through an AI tutor that can teach anything, help with programming, writing, analysis, thinking, and reasoning. \n* Making people "super humans" not through supernatural abilities, but by providing them with "super AIs."'